# Change of Variables (Probability)

**Concept 28** of the math-foundations map. Prereqs: `24-pdf`, `14-gradient-jacobian`.

If $Y = g(X)$ for a $C^1$ bijection $g$, then
$$f_Y(y) = f_X(g^{-1}(y))\,|\det J_{g^{-1}}(y)|.$$
We verify this numerically with $X\sim\mathrm{Uniform}[0,1]$, $g(x) = -\log x$, predicting $Y\sim\mathrm{Exp}(1)$.


In [ ]:
import math, random
random.seed(20260514)

N = 10_000
xs = [random.random() for _ in range(N)]
ys = [-math.log(x) for x in xs if x > 0.0]
print(f"N = {len(ys)}, mean(Y) = {sum(ys)/len(ys):.4f}  (analytic 1.0)")


## 1. Empirical PDF of $Y$

Histogram on $[0, 6]$ with 60 bins. Compare to $e^{-y}$.

In [ ]:
def empirical_pdf(samples, lo, hi, n_bins):
    width = (hi - lo) / n_bins
    counts = [0]*n_bins
    for s in samples:
        if lo <= s < hi:
            counts[int((s - lo)/width)] += 1
    n = len(samples)
    centers = [lo + (k+0.5)*width for k in range(n_bins)]
    dens = [c/(n*width) for c in counts]
    return centers, dens

centers, dens = empirical_pdf(ys, 0.0, 6.0, 60)
print(f"{'y':>6} {'empirical':>10} {'e^{-y}':>10} {'|err|':>8}")
for y, d in zip(centers, dens):
    if abs(y - round(y*2)/2) < 1e-9 and y <= 4.5:
        a = math.exp(-y)
        print(f"{y:>6.2f} {d:>10.4f} {a:>10.4f} {abs(d-a):>8.4f}")


## 2. ASCII plot of histogram vs analytic

Each row is a bin; bar length is proportional to density.

In [ ]:
width_chars = 40
y_max = max(dens) if dens else 1.0
for y, d in zip(centers, dens):
    if y > 4.0: break
    bar = '#' * int(round(d/y_max * width_chars))
    star = '*' * int(round(math.exp(-y)/y_max * width_chars))
    print(f"y={y:4.2f} emp |{bar:<40}|  ana |{star:<40}|")


## 3. Pointwise check of the CoV formula

For $g(x) = -\log x$ on $(0,1)$: $g^{-1}(y) = e^{-y}$, $|g'(x)| = 1/x$.
So $f_Y(y) = f_X(e^{-y}) / (1/e^{-y}) = e^{-y}$ since $f_X \equiv 1$ on $(0,1)$.

In [ ]:
f_X = 1.0
print(f"{'y':>6} {'x=g^-1(y)':>10} {'|g_prime|':>10} {'predicted':>10} {'e^{-y}':>10}")
for y in [0.1, 0.5, 1.0, 2.0, 3.0]:
    x = math.exp(-y)
    g_prime_abs = 1.0/x
    pred = f_X / g_prime_abs
    ana = math.exp(-y)
    print(f"{y:>6.2f} {x:>10.4f} {g_prime_abs:>10.4f} {pred:>10.6f} {ana:>10.6f}")
    assert abs(pred - ana) < 1e-12
print("\nOK: formula verified pointwise.")


## 4. Sanity: inverse-CDF sampling reproduces the same thing

The Exp(1) CDF is $F(y)=1-e^{-y}$, so $F^{-1}(u) = -\log(1-u)$. Sampling $U\sim\mathrm{Uniform}[0,1]$ and returning $F^{-1}(U)$ should also give Exp(1). Note $-\log(1-U)$ has the same distribution as $-\log U$ since $1-U \sim U$.

In [ ]:
us = [random.random() for _ in range(N)]
ys2 = [-math.log(1.0 - u) for u in us if u < 1.0]
print(f"mean via inverse-CDF: {sum(ys2)/len(ys2):.4f}   (analytic 1.0)")
print(f"mean via Y = -log X : {sum(ys)/len(ys):.4f}   (analytic 1.0)")


## Summary

- Change of variables in 1D: $f_Y(y) = f_X(g^{-1}(y)) / |g'(g^{-1}(y))|$.
- Multivariate: replace $1/|g'|$ with $|\det J_{g^{-1}}|$.
- This formula is the backbone of normalizing flows (sum of $\log|\det J|$ per layer) and inverse-CDF sampling, and the infinitesimal form $\partial_t \log p_t = -\nabla\cdot v_t$ powers continuous-time flow / ELF-style velocity-field training.